# Export vocoloco du narrateur (prêt pour NekoSpeak)

Prend `lora_narrateur_adapter.zip` (sortie du notebook d'entraînement) et produit
**les fichiers exacts que l'app NekoSpeak sait charger** :

1. **Fusion** des poids LoRA dans k2-fsa/OmniVoice (modèle complet fp32) ;
2. **Export ONNX "vocoloco"** : le graphe principal `omnivoice-main-split.onnx`
   avec entrées `input_ids (B,8,S)` / `audio_mask (B,S)` / `attention_mask (B,1,S,S)`
   et sortie `audio_logits (B,8,S,1025)` — mêmes noms, mêmes rangs, mêmes types que
   `Gigsu/vocoloco-onnx` (le format consommé par OmniVoiceEngine.kt) ;
3. **Validation bout-en-bout** : génération complète (20 pas de diffusion) en PyTorch
   puis en ONNX pur, avec l'encodeur et le décodeur Higgs **ONNX officiels de l'app**
   (identiques à ceux téléchargés par NekoSpeak) ;
4. Zip dans **Mon Drive/omnivoice/** + upload optionnel vers Hugging Face.

## Pourquoi ce format et pas l'int4 onnx-community ?
L'app utilise le moteur vocoloco (`Gigsu/vocoloco-onnx`), pas la chaîne onnx-community.
On remplace uniquement les poids du backbone (les 2 fichiers principaux) ; le décodeur,
l'encodeur de clonage, le tokenizer et la config restent ceux d'origine.

## Avant de lancer
- `lora_narrateur_adapter.zip` dans ton Drive (retrouvé automatiquement) ;
- un GPU (T4 suffit) rend la validation ~5x plus rapide : garde l'onglet ouvert ;
- durée totale ~45-75 min (fusion ~15 min, export ~15 min, validations ~15 min).

> Licence : poids OmniVoice **CC BY-NC** -> hébergement HF **privé** recommandé.

## 0. Montage du Drive et configuration

In [ ]:
# ===== CONFIGURATION =====
import os, glob, json, shutil, math, sys
import torch
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive'
DRIVE_OUT  = os.path.join(DRIVE_ROOT, 'omnivoice')
os.makedirs(DRIVE_OUT, exist_ok=True)

WORK       = '/content/work'
MERGED_DIR = os.path.join(WORK, 'narrateur-merge')
EXPORT_DIR = os.path.join(WORK, 'vocoloco_narrateur')
VC_DIR     = os.path.join(WORK, 'gigsu_onnx')   # encodeur/décodeur Higgs officiels de l'app
os.makedirs(WORK, exist_ok=True)

TEXT_TEST = "La brume rampait entre les pierres noires de la nécropole, et nul oiseau ne chantait."
# Transcription EXACTE de la référence (clip_0733, 10,3 s — cf. train.jsonl du dataset)
REF_TEXT_HARD = ("Il déambula dans les rues désertes et les jardins publics d'Agrillon, "
                 "réfléchissant avec gravité aux propos de l'estropier et le maudissant, "
                 "sans pouvoir néanmoins mettre cette conversation de côté.")

print('GPU :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'AUCUN (CPU, plus lent mais OK)')

# --- recherche de l'adaptateur et de la référence dans tout MyDrive ---
ADAPTER_ZIP = REF_WAV = REF_TXT = None
for root, dirs, files in os.walk(DRIVE_ROOT):
    dirs[:] = [d for d in dirs if not d.startswith('.')]
    if root[len(DRIVE_ROOT):].count(os.sep) > 8:
        dirs[:] = []; continue
    for f in files:
        if f == 'lora_narrateur_adapter.zip' and ADAPTER_ZIP is None:
            ADAPTER_ZIP = os.path.join(root, f)
        if f == 'ref_narrateur.wav' and REF_WAV is None:
            REF_WAV = os.path.join(root, f)
        if f == 'ref_narrateur.txt' and REF_TXT is None:
            REF_TXT = os.path.join(root, f)

if ADAPTER_ZIP is None:
    raise FileNotFoundError('lora_narrateur_adapter.zip introuvable dans MyDrive.')
print('Adaptateur :', ADAPTER_ZIP)
print('Ref wav    :', REF_WAV)
print('Ref txt    :', REF_TXT)

# --- secours 1 : chercher clip_0733.wav dans les zips du dataset du Drive ---
# (clip_0733 = la référence d'entraînement v4 ; on ÉVITE ref_narrateur.wav des
#  datasets v1-v3 qui est l'ancienne référence, ~12 s, remplacée en v4)
if REF_WAV is None or (os.path.basename(REF_WAV) == 'ref_narrateur.wav'):
    import zipfile as _zf
    _refdir = os.path.join(WORK, 'ref'); os.makedirs(_refdir, exist_ok=True)
    _zips = []
    for root, dirs, files in os.walk(DRIVE_ROOT):
        dirs[:] = [d for d in dirs if not d.startswith('.')]
        for f in files:
            if f.endswith('.zip') and 'dataset' in f.lower() and 'narrateur' in f.lower():
                _zips.append(os.path.join(root, f))
    for z in sorted(_zips, key=lambda p: 'v4' in p.lower(), reverse=True):
        with _zf.ZipFile(z) as zf:
            tgt = next((n for n in zf.namelist() if n.endswith('clip_0733.wav')), None)
            if tgt is None: continue
            zf.extract(tgt, _refdir)
            REF_WAV = os.path.join(_refdir, tgt)
            print('Référence extraite de', z)
            break

# --- secours 2 : asset public de la branche Kotlin (clip_0733 validé, 493 Ko) ---
if REF_WAV is None or (os.path.basename(REF_WAV) == 'ref_narrateur.wav'
                       and os.path.getsize(REF_WAV) != 493484):
    pass  # déjà trouvé ailleurs
if REF_WAV is None:
    import urllib.request
    REF_WAV = os.path.join(WORK, 'narrateur_ref.wav')
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/kalorg/NekoSpeak/feat/omnivoice-narrateur/'
        'app/src/main/assets/narrateur_ref.wav', REF_WAV)
    print('Référence téléchargée depuis la branche NekoSpeak (GitHub)')

# --- contrôle de la référence : clip_0733 fait 10,28 s (l'ancienne fait ~12 s) ---
import wave as _w
with _w.open(REF_WAV) as _wf:
    _dur = _wf.getnframes() / _wf.getframerate()
    print(f'Référence : {_dur:.2f}s, {_wf.getframerate()} Hz, {_wf.getnchannels()} canal(s)')
assert 9.0 < _dur < 11.5, 'durée inattendue : ce n est probablement pas clip_0733'


## 1. Installation (~5 min)

`omnivoice` depuis GitHub (fusion LoRA + code du modèle), puis `onnx` + `onnxruntime`
pour l'export et la validation Android-fidèle.

In [ ]:
%pip install -q "omnivoice @ git+https://github.com/k2-fsa/OmniVoice.git" peft
%pip install -q onnx onnxruntime soundfile
# Colab préinstalle torchao 0.10.0, incompatible avec peft >= 1.0 : le chargement
# de l'adaptateur LoRA échoue (PEFT exige torchao >= 0.16... ou aucun torchao).
# Notre LoRA est standard -> on le désinstalle carrément.
%pip uninstall -y -q torchao


In [ ]:
# Contrôles : la fusion LoRA et la conversion ONNX sont bien disponibles
import subprocess, sys
r = subprocess.run([sys.executable, '-c', 'import omnivoice; print(omnivoice.__file__)'],
                  capture_output=True, text=True)
print('omnivoice :', r.stdout.strip() or r.stderr.strip())
r = subprocess.run(['omnivoice-merge-lora', '--help'], capture_output=True, text=True)
ok = '--lora_adapter' in (r.stdout + r.stderr)
print('omnivoice-merge-lora LoRA :', 'OK' if ok else 'ABSENT — relance la cellule d\'install')
assert ok, "fusion LoRA indisponible"
import onnx, onnxruntime, transformers, peft
print('onnx', onnx.__version__, '| ort', onnxruntime.__version__,
      '| transformers', transformers.__version__, '| peft', peft.__version__)

## 2. Déballage de l'adaptateur et de la référence vocale

L'adaptateur ne fait que ~110 Mo (adapter_config.json + adapter_model.safetensors) ;
il encapsule les deltas LoRA appris sur ta voix de narrateur.

In [ ]:
import zipfile
ADAPTER_DIR = os.path.join(WORK, 'adapter')
if os.path.isdir(ADAPTER_DIR) and os.path.exists(os.path.join(ADAPTER_DIR, 'adapter_config.json')):
    print('Adaptateur déjà déballé')
else:
    with zipfile.ZipFile(ADAPTER_ZIP) as z:
        z.extractall(ADAPTER_DIR)
    # si le zip contenait un sous-dossier
    if not os.path.exists(os.path.join(ADAPTER_DIR, 'adapter_config.json')):
        for root, _, files in os.walk(ADAPTER_DIR):
            if 'adapter_config.json' in files:
                ADAPTER_DIR = root; break
print(json.dumps(json.load(open(os.path.join(ADAPTER_DIR, 'adapter_config.json'))), indent=1)[:600])

REF_TEXT = REF_TEXT_HARD
if REF_TXT and os.path.exists(REF_TXT):
    REF_TEXT = open(REF_TXT, encoding='utf-8').read().strip()
print('Texte de ref utilisé :', REF_TEXT[:80], '...')
if REF_WAV is None:
    raise FileNotFoundError("ref_narrateur.wav introuvable — il faut la référence clip_0733 "
                            "(elle est dans Mon Drive/omnivoice/ ou dans le zip du dataset).")

## 3. Fusion LoRA -> modèle complet (~15 min)

`omnivoice-merge-lora` absorbe les deltas LoRA dans les poids de base.
Résultat : un modèle HF autonome (~2,5 Go), directement exportable.

In [ ]:
import subprocess
if os.path.exists(os.path.join(MERGED_DIR, 'config.json')):
    print('Merge déjà présent, on passe.')
else:
    cmd = ['omnivoice-merge-lora', '--base_model', 'k2-fsa/OmniVoice',
           '--lora_adapter', ADAPTER_DIR, '--output_dir', MERGED_DIR]
    print('>>', ' '.join(cmd), flush=True)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end='', flush=True)
    p.wait()
    assert p.returncode == 0, f"fusion echouee (code {p.returncode})"
print('Fichiers merge :', sorted(os.listdir(MERGED_DIR))[:12], '...')
assert os.path.exists(os.path.join(MERGED_DIR, 'audio_tokenizer', 'config.json')) or \
       os.path.exists(os.path.join(MERGED_DIR, 'audio_tokenizer')), 'audio_tokenizer absent du merge'
print('OK — narrateur-merge prêt')

## 4. Wrapper « vocoloco » + sessions ONNX officielles Higgs

On enrobe le modèle exactement comme `Gigsu/vocoloco-onnx` : entrées
`input_ids (B,8,S)` + `audio_mask (B,S)` + `attention_mask (B,1,S,S) bool`, sortie
`audio_logits (B,8,S,1025)`. On télécharge aussi l'encodeur et le décodeur Higgs
OFFICIELS de l'app (654 Mo + 83 Mo) pour une validation strictement identique
à ce que fera le téléphone.

In [ ]:
from transformers import AutoModel
import omnivoice  # enregistre le model_type dans transformers
import torch.nn as nn

model = AutoModel.from_pretrained(MERGED_DIR, trust_remote_code=True,
                                  torch_dtype=torch.float32, attn_implementation='eager')
model.eval().to('cuda' if torch.cuda.is_available() else 'cpu')
DEV = next(model.parameters()).device

for a in ['llm', 'audio_embeddings', 'audio_heads', 'codebook_layer_offsets']:
    print(f'  attr {a:24s}:', 'OUI' if hasattr(model, a) else 'NON')
assert hasattr(model, 'audio_embeddings') and hasattr(model, 'audio_heads')

# NB : model.llm est déjà le Qwen3Model NU (AutoModel.from_config dans
# omnivoice.py) — pas de .model imbriqué à déballer.
backbone = model.llm
print('backbone :', type(backbone).__name__, '| hidden :', backbone.config.hidden_size)

class VocolocoMain(nn.Module):
    """input_ids (B,C,S) int64 | audio_mask (B,S) bool | attention_mask (B,1,S,S) bool
       -> audio_logits (B,C,S,V) float32. Reproduit _prepare_embed_inputs + llm + heads."""
    def __init__(self, m):
        super().__init__()
        self.text_embed = m.get_input_embeddings()
        self.audio_embed = m.audio_embeddings
        self.register_buffer('layer_offsets',
                             torch.as_tensor(m.codebook_layer_offsets, dtype=torch.long).clone())
        self.backbone = m.llm
        self.heads = m.audio_heads
    def forward(self, input_ids, audio_mask, attention_mask):
        text_e = self.text_embed(input_ids[:, 0, :])
        shifted = input_ids * audio_mask.unsqueeze(1).to(input_ids.dtype) + \
                  self.layer_offsets.view(1, -1, 1)
        audio_e = self.audio_embed(shifted).sum(dim=1)
        embeds = torch.where(audio_mask.unsqueeze(-1), audio_e, text_e)
        neg = torch.finfo(embeds.dtype).min
        add = (1.0 - attention_mask.to(embeds.dtype)) * neg   # True=garde, False=-max
        hs = self.backbone(inputs_embeds=embeds, attention_mask=add).last_hidden_state
        B, S, H = hs.shape
        return self.heads(hs).view(B, S, 8, 1025).permute(0, 2, 1, 3)

wrapper = VocolocoMain(model).eval().to(DEV)
with torch.no_grad():
    B, S = 2, 48
    d_ids = torch.zeros(B, 8, S, dtype=torch.long, device=DEV)
    d_am  = torch.zeros(B, S, dtype=torch.bool, device=DEV); d_am[:, 24:] = True
    d_at  = torch.ones(B, 1, S, S, dtype=torch.bool, device=DEV)
    out = wrapper(d_ids, d_am, d_at)
print('wrapper OK, logits :', tuple(out.shape))
n_params = sum(p.numel() for p in wrapper.parameters())
print(f'paramètres backbone wrapper : {n_params/1e6:.0f} M_param')

In [ ]:
# Sessions ONNX officielles (celles de l'app)
from huggingface_hub import hf_hub_download
import onnxruntime as ort
os.makedirs(VC_DIR, exist_ok=True)
enc_path = hf_hub_download('Gigsu/vocoloco-onnx', 'omnivoice-encoder-fixed.onnx', local_dir=VC_DIR)
dec_path = hf_hub_download('Gigsu/vocoloco-onnx', 'omnivoice-decoder.onnx', local_dir=VC_DIR)
opts = ort.SessionOptions(); opts.intra_op_num_threads = 4
enc_sess = ort.InferenceSession(enc_path, sess_options=opts, providers=['CPUExecutionProvider'])
dec_sess = ort.InferenceSession(dec_path, sess_options=opts, providers=['CPUExecutionProvider'])
print('encodeur I/O :', [(i.name, i.shape) for i in enc_sess.get_inputs()],
      [(o.name, o.shape) for o in enc_sess.get_outputs()])
print('décodeur I/O :', [(i.name, i.shape) for i in dec_sess.get_inputs()],
      [(o.name, o.shape) for o in dec_sess.get_outputs()])

## 5. Boucle de génération (portage exact du moteur Android)

Portage Python 1:1 d'OmniVoiceEngine.kt : masques, PRNG seedé (mulberry32),
planning de démasquage, CFG (guidance 4.0), pénalité de couche 5.0, température 5.0,
top-k unmasking, décodage Higgs ONNX. Le même code servira pour la validation
PyTorch puis pour la validation ONNX.

In [ ]:
import numpy as np, soundfile as sf
from transformers import AutoTokenizer

tokz = AutoTokenizer.from_pretrained(MERGED_DIR)
SP = dict(denoise=151669, lang_start=151670, lang_end=151671, inst_start=151672,
          inst_end=151673, text_start=151674, text_end=151675)
C, V, MASK = 8, 1025, 1024

def np_encode(text):
    return tokz(text, add_special_tokens=False)['input_ids']

# CORRECTIF v2 : quand une valeur manque, le processor d'entraînement met le
# mot "None" en toutes lettres entre les marqueurs (instruct_ratio=0.0 dans
# notre config => <|instruct_start|>None<|instruct_end|> a 100% du training).
# denoise=False = variant la plus proche de la distribution d'entraînement
# (le dataset v4 n'a aucun sample de prompt-denoising).
def style_ids(lang='fr', instruct='None', denoise=True):
    ids = [SP['denoise']] if denoise else []
    ids += [SP['lang_start']] + np_encode(lang) + [SP['lang_end'], SP['inst_start']]
    ids += np_encode(instruct) + [SP['inst_end']]
    return ids

def text_with_ref_ids(sentence, ref_text):
    import re
    full = (ref_text.strip() + ' ' + sentence.strip()) if ref_text else sentence.strip()
    full = re.sub(r'[\r\n]+', '', full); full = re.sub(r'[ \t]+', ' ', full)
    return [SP['text_start']] + np_encode(full) + [SP['text_end']]

def mulberry32(seed):
    s = seed & 0xFFFFFFFF
    def rnd():
        nonlocal s
        s = (s + 0x6D2B79F5) & 0xFFFFFFFF
        t = ((s ^ (s >> 15)) * (1 | s)) & 0xFFFFFFFF
        t = (t + ((t ^ (t >> 7)) * (61 | t)) & 0xFFFFFFFF) & 0xFFFFFFFF
        u = t ^ (t >> 14)
        return u / 4294967296.0
    return rnd

def time_steps(n, t_shift=0.05):
    ts = []
    for i in range(n + 1):
        t = i / n
        ts.append(t_shift * t / (1 + (t_shift - 1) * t))
    return ts

def schedule_for(total, n):
    ts = time_steps(n)
    sched, remaining = [], total
    for s in range(n):
        frac = ts[s + 1] - ts[s] if s + 1 < len(ts) else 0.0
        k = min(math.ceil(total * frac), remaining)
        if s == n - 1: k = remaining
        sched.append(k); remaining -= k
    return sched

def phonetic_weight(text):
    w = 0.0
    for ch in text:
        o = ord(ch)
        if ch.isspace(): w += 0.2
        elif ch.isdigit(): w += 3.5
        elif 0x2E80 <= o <= 0x9FFF or 0xAC00 <= o <= 0xD7AF: w += 3.0
        elif ch.isalpha(): w += 1.0
        else: w += 0.5
    return w

def estimate_target_tokens(sentence, ref_text, ref_T):
    wr = max(phonetic_weight(ref_text), 1.0)
    tpw = ref_T / wr
    return max(1, round(phonetic_weight(sentence) * tpw))

def encode_ref_onnx(wav_path):
    wav, sr = sf.read(wav_path, dtype='float32')
    if wav.ndim > 1: wav = wav[:, 0]
    assert abs(sr - 24000) < 1, f'reference a {sr} Hz, il faut 24 kHz mono'
    codes = enc_sess.run(['audio_codes'], {'input_values': wav[None, None, :]})[0]
    return codes[0].astype(np.int64)  # (8, T)

REF_TOKENS = encode_ref_onnx(REF_WAV)
REF_T = REF_TOKENS.shape[1]
print(f'Reference encodee : codebooks={REF_TOKENS.shape[0]}, frames={REF_T} '
      f'({REF_T/75:.1f}s @75trs/s), max id={REF_TOKENS.max()}')
assert REF_TOKENS.max() < MASK


In [ ]:
# --- preparation des entrees (portage de prepareInferenceInputs.kt) ---
def build_inputs(style, text_ids, n_target, ref_tokens=None):
    ref_len = ref_tokens.shape[1] if ref_tokens is not None else 0
    total = len(style) + len(text_ids) + ref_len + n_target
    ids = np.full((2, C, total), MASK, dtype=np.int64)
    for c in range(C):
        pos = 0
        for t in style: ids[0, c, pos] = t; pos += 1
        for t in text_ids: ids[0, c, pos] = t; pos += 1
        if ref_tokens is not None:
            ids[0, c, pos:pos+ref_len] = ref_tokens[c]; pos += ref_len
        # cible : reste masquee
        for t in range(n_target): ids[1, c, t] = MASK
    audio_mask = np.zeros((2, total), dtype=bool)
    a_start = len(style) + len(text_ids)
    audio_mask[0, a_start:] = True
    audio_mask[1, :n_target] = True
    attn = np.zeros((2, 1, total, total), dtype=bool)
    attn[0, 0, :total, :total] = True            # cond : attention pleine
    attn[1, 0, :n_target, :n_target] = True     # uncond : zone cible seule
    for p in range(n_target, total): attn[1, 0, p, p] = True
    return dict(input_ids=ids, audio_mask=audio_mask, attention_mask=attn), total, n_target, \
           a_start + ref_len

def log_softmax(x, axis=-1):
    m = x.max(axis=axis, keepdims=True)
    y = x - m
    return y - np.log(np.exp(y).sum(axis=axis, keepdims=True))

def logsumexp(x):
    m = x.max()
    return m + np.log(np.exp(x - m).sum())

# CORRECTIF v2 de la boucle : la formule officielle est
#   combined = log_softmax(c) + g * (log_softmax(c) - log_softmax(u))
#   lp       = log_softmax(combined)         <-- SOUS-TRAIRE logsumexp, PAS
#                                              soustraire le vecteur log_softmax !
# L'ancienne version produisait un vecteur constant => argmax=0 partout =>
# tous les codes audio a 0, d'ou l'audio catastrophic.
def generate(loop_forward, sentence, ref_text, ref_tokens, seed=42, num_steps=20,
             guidance=4.0, layer_penalty=5.0, pos_temp=5.0, lang='fr',
             instruct='None', denoise=True, verbose=True):
    style = style_ids(lang=lang, instruct=instruct, denoise=denoise)
    tids = text_with_ref_ids(sentence, ref_text)
    n_target = estimate_target_tokens(sentence, ref_text, ref_tokens.shape[1])
    inputs, total, n_t, t_off = build_inputs(style, tids, n_target, ref_tokens)
    ids = inputs['input_ids'].copy()
    sched = schedule_for(C * n_t, num_steps)
    rng = mulberry32(seed)
    tokens = np.full(C * n_t, MASK, dtype=np.int64)
    g1 = 1.0 + guidance
    for step, k in enumerate(sched):
        if k <= 0: continue
        ids_work = ids.copy()
        cur = tokens.reshape(C, n_t)
        for c in range(C):
            ids_work[0, c, t_off:t_off+n_t] = cur[c]
            ids_work[1, c, :n_t] = cur[c]
        logits = loop_forward(ids_work, inputs['audio_mask'], inputs['attention_mask'])
        pred = np.zeros(C * n_t, dtype=np.int64)
        score = np.zeros(C * n_t)
        for c in range(C):
            for t in range(n_t):
                cl = logits[0, c, t_off + t]
                ul = logits[1, c, t]
                lc, lu = log_softmax(cl), log_softmax(ul)
                combined = g1 * lc - guidance * lu      # == c + g*(c - u)
                lp = combined - logsumexp(combined)     # log_softmax final (officiel)
                lp = lp.copy(); lp[MASK] = -np.inf      # mask_id jamais predit
                j = int(lp.argmax())
                pred[c * n_t + t] = j
                score[c * n_t + t] = lp[j] - layer_penalty * c
        noise = np.array([-math.log(-math.log(rng() + 1e-10) + 1e-10) for _ in range(C * n_t)])
        open_pos = tokens == MASK
        score = np.where(open_pos, score / pos_temp + noise, -np.inf)
        order = np.argsort(-score)
        for i in order[:k]:
            tokens[i] = pred[i]
        if verbose:
            print(f'  pas {step+1}/{num_steps} : {k} jetons demasques', end='\r', flush=True)
    if verbose: print()
    codes = tokens.reshape(C, n_t)[None, :, :]          # (1,8,T)
    audio = dec_sess.run(['audio_values'], {'audio_codes': codes.astype(np.int64)})[0]
    return postprocess(audio.reshape(-1))

def postprocess(pcm):
    thresh, margin = 0.005, int(24000 * 0.02)
    nz = np.abs(pcm) > thresh
    if not nz.any(): return pcm
    s = max(0, int(np.argmax(nz)) - margin)
    e = min(len(pcm), len(pcm) - int(np.argmax(nz[::-1])) + margin)
    out = pcm[s:e].copy()
    peak = np.abs(out).max()
    if peak > 1e-6: out *= 0.5 / peak
    return out

def torch_forward(ids, am, at):
    with torch.no_grad():
        t = lambda x: torch.from_numpy(x).to(DEV)
        return wrapper(t(ids), t(am), t(at)).float().cpu().numpy()

import IPython.display as ipd

# --- Ecoute : 3 generations PyTorch a comparer ---
wav_A = generate(torch_forward, TEXT_TEST, REF_TEXT, REF_TOKENS, lang='fr', denoise=True)
sf.write(os.path.join(WORK, 'narrateur_pytorch_denoise.wav'), wav_A, 24000)
print('A/ denoise + fr (variante officielle)')
ipd.display(ipd.Audio(wav_A, rate=24000))

wav_B = generate(torch_forward, TEXT_TEST, REF_TEXT, REF_TOKENS, lang='fr', denoise=False)
sf.write(os.path.join(WORK, 'narrateur_pytorch_sans_denoise.wav'), wav_B, 24000)
print('B/ sans denoise + fr (distribution d\'entrainement, instruct "None")')
ipd.display(ipd.Audio(wav_B, rate=24000))

# --- Reference absolue : generate() officiel du modele fusionne ---
from omnivoice import OmniVoiceGenerationConfig
gc = OmniVoiceGenerationConfig(num_step=20, guidance_scale=4.0, t_shift=0.05)
wav_O = model.generate(TEXT_TEST, language='fr', ref_audio=REF_WAV,
                       ref_text=REF_TEXT, generation_config=gc)[0]
sf.write(os.path.join(WORK, 'narrateur_officiel_fr.wav'), wav_O, 24000)
print('O/ generation 100% officielle (temoin du merge LoRA)')
ipd.display(ipd.Audio(wav_O, rate=24000))


## 6. Export ONNX fp32 « vocoloco » (~15 min)

Le graphe est exporté SANS poids (petit fichier), puis les ~2,45 Go de poids fp32
sont attachés comme **données externes** dans un fichier unique
`omnivoice-main.onnx_data`, exactement comme Gigsu : on évite ainsi la limite protobuf de 2 Go.

In [ ]:
raw_graph = os.path.join(WORK, 'main_raw.onnx')
os.makedirs(EXPORT_DIR, exist_ok=True)
GRAPH_OUT = os.path.join(EXPORT_DIR, 'omnivoice-main-split.onnx')
DATA_FILE = 'omnivoice-main.onnx_data'

# FIX v3 : do_constant_folding=False OBLIGATOIRE avec export_params=False :
# le repliage transforme chaque Transpose(poids) des nn.Linear en constante
# anonyme (onnx::MatMul_xxxx)_innommable et inattachable.
# (Les buffers non persistants etaient la 1re hypothese, vite ecartee.)
# (inv_freq du rotary Qwen3, coefficients d'echelle...). export_params=False
# nomme les entrees d'apres le state_dict, qui ignore justement ces buffers ->
# ils ressortent avec des noms generes, inattachables. On les re-declare
# persistants : l'exportateur et le state_dict les nomment alors pareil.
# (Le DeprecationWarning "legacy TorchScript-based ONNX export" est attendu.)
wrapper_cpu = wrapper.cpu().eval()
made_persistent = []
for name, mod in wrapper_cpu.named_modules():
    npp = getattr(mod, '_non_persistent_buffers_set', set())
    for bname in list(getattr(mod, '_buffers', {}).keys()):
        if bname in npp:
            mod.register_buffer(bname, getattr(mod, bname).detach().clone(),
                                persistent=True)
            made_persistent.append(f'{name}.{bname}' if name else bname)
print('buffers re-declares persistants :', made_persistent or 'aucun')

B, S = 2, 64
d_ids = np.zeros((B, 8, S), dtype=np.int64)
d_ids[:, :, 40:] = 123
d_am = np.zeros((B, S), dtype=bool); d_am[:, 40:] = True
d_at = np.ones((B, 1, S, S), dtype=bool)
torch.onnx.export(
    wrapper_cpu, (torch.from_numpy(d_ids), torch.from_numpy(d_am), torch.from_numpy(d_at)),
    raw_graph,
    input_names=['input_ids', 'audio_mask', 'attention_mask'],
    output_names=['audio_logits'],
    dynamic_axes={'input_ids': {0: 'batch', 2: 'seq_len'},
                  'audio_mask': {0: 'batch', 1: 'seq_len'},
                  'attention_mask': {0: 'batch', 2: 'seq_len', 3: 'seq_len'},
                  'audio_logits': {0: 'batch', 2: 'seq_len'}},
    opset_version=17, do_constant_folding=False, export_params=False,
    dynamo=False)  # exporteur TorchScript : torch>=2.8 route sinon vers onnxscript absent

m = onnx.load(raw_graph)
keep = {'input_ids', 'audio_mask', 'attention_mask'}
weight_names = [i.name for i in m.graph.input if i.name not in keep]
init_names = {t.name for t in m.graph.initializer}
print('poids attendus dans le graphe :', len(weight_names),
      '| buffers inlines :', len(init_names))

sd = {k: v.detach().cpu().contiguous() for k, v in wrapper_cpu.state_dict().items()}
missing = [n for n in weight_names if n not in sd]
if missing:
    meta = {i.name: i.type.tensor_type for i in m.graph.input if i.name not in keep}
    detail = [(n, onnx.helper.tensor_dtype_to_string(meta[n].elem_type),
               tuple(d.dim_value for d in meta[n].shape.dim)) for n in missing[:10]]
    raise AssertionError(f'{len(missing)} poids introuvables dans le state_dict : {detail}')

DTYPES = {np.dtype('float32'): onnx.TensorProto.FLOAT,
          np.dtype('float64'): onnx.TensorProto.DOUBLE,
          np.dtype('int64'):  onnx.TensorProto.INT64,
          np.dtype('int32'):  onnx.TensorProto.INT32}
blob_path = os.path.join(EXPORT_DIR, DATA_FILE)
offset = 0
with open(blob_path, 'wb') as f:
    for n in weight_names:
        arr = sd[n].numpy()
        assert arr.dtype in DTYPES, f'{n}: dtype inattendu {arr.dtype}'
        b = arr.tobytes()
        t = onnx.TensorProto()
        t.name = n; t.data_type = DTYPES[arr.dtype]
        t.dims.extend(list(arr.shape))
        t.data_location = onnx.TensorProto.EXTERNAL
        for kkey, vval in [('location', DATA_FILE), ('offset', str(offset)), ('length', str(len(b)))]:
            e = t.external_data.add(); e.key = kkey; e.value = vval
        m.graph.initializer.append(t)
        f.write(b)
        offset += len(b)
        del arr, b
del sd

# ne garder que les 3 vraies entrees, dans l'ordre attendu par l'app
kept = {i.name: i for i in m.graph.input if i.name in keep}
assert len(kept) == 3, f'entrees reelles manquantes : {keep - set(kept)}'
del m.graph.input[:]
for n in ['input_ids', 'audio_mask', 'attention_mask']:
    m.graph.input.append(kept[n])
onnx.save(m, GRAPH_OUT)
print('graphe :', GRAPH_OUT, f'({os.path.getsize(GRAPH_OUT)/1e6:.1f} Mo)')
print('donnees :', f'{offset/1e9:.2f} Go')


## 6bis. Quantification INT8 dynamique (recommandée pour Android CPU)

Le backbone FP32 (~2,4 Go) nécessite ~38 Go de lecture RAM par phrase sur téléphone
(~50–75 s/phrase). La quantification dynamique **INT8** réduit `omnivoice-main.onnx_data`
à **~600 Mo** (3 à 4× plus rapide sur CPU ARM64). Passez `QUANTIZE_INT8 = False` pour
conserver le FP32 d'origine.

In [ ]:
QUANTIZE_INT8 = True

if QUANTIZE_INT8:
    from onnxruntime.quantization import quantize_dynamic, QuantType

    fp32_graph = os.path.join(EXPORT_DIR, "omnivoice-main-fp32.onnx")
    fp32_data = os.path.join(EXPORT_DIR, "omnivoice-main-fp32.onnx_data")
    q_raw = os.path.join(WORK, "main_int8_raw.onnx")

    # Renomme le graphe FP32 en entree temporaire
    shutil.move(GRAPH_OUT, fp32_graph)
    shutil.move(blob_path, fp32_data)
    # Met a jour la reference externe dans fp32_graph
    m_fp32 = onnx.load(fp32_graph, load_external_data=False)
    for init in m_fp32.graph.initializer:
        for entry in init.external_data:
            if entry.key == "location":
                entry.value = "omnivoice-main-fp32.onnx_data"
    onnx.save(m_fp32, fp32_graph)

    print("Quantification dynamique INT8 en cours (~1-2 min)...")
    quantize_dynamic(
        model_input=fp32_graph,
        model_output=q_raw,
        weight_type=QuantType.QInt8,
        use_external_data_format=True,
    )

    # Re-empaquette exactement sous omnivoice-main-split.onnx + omnivoice-main.onnx_data
    m_q = onnx.load(q_raw, load_external_data=True)
    onnx.save_model(
        m_q, GRAPH_OUT,
        save_as_external_data=True,
        all_tensors_to_one_file=True,
        location=DATA_FILE,
        size_threshold=1024,
    )
    for tmp in (fp32_graph, fp32_data, q_raw):
        if os.path.exists(tmp):
            os.remove(tmp)
    print(f"OK INT8 : {os.path.getsize(GRAPH_OUT)/1e6:.2f} Mo (graphe) + "
          f"{os.path.getsize(blob_path)/1e6:.1f} Mo (poids)")


## 7. Validation ONNX (identique au téléphone)

1. Écart max de logits wrapper PyTorch vs session ONNX (doit être < 1e-3) ;
2. Génération complète ONNX en `fr` et en `en` (l'app envoie `en` pour le français :
   on écoute les deux avant de choisir) ;
3. Similarité cosinus avec la génération PyTorch (même graine).

In [ ]:
import IPython.display as ipd

main_sess = ort.InferenceSession(GRAPH_OUT, sess_options=opts, providers=['CPUExecutionProvider'])
ios = [(i.name, i.type, i.shape) for i in main_sess.get_inputs()]
print('ENTREES :'); [print('  ', n, t, s) for n, t, s in ios]
print('SORTIES :'); [print('  ', o.name, o.type, o.shape) for o in main_sess.get_outputs()]
assert [n for n, _, _ in ios] == ['input_ids', 'audio_mask', 'attention_mask']

with torch.no_grad():
    ref_logits = wrapper_cpu(torch.from_numpy(d_ids), torch.from_numpy(d_am),
                             torch.from_numpy(d_at)).numpy()
ort_logits = main_sess.run(None, {'input_ids': d_ids, 'audio_mask': d_am,
                                  'attention_mask': d_at})[0]
diff = np.abs(ort_logits - ref_logits).max()
rel = diff / max(1e-9, np.abs(ref_logits).max())
print(f'ecart max logits : {diff:.2e} (relatif {rel:.2e})')
tol = 0.25 if globals().get('QUANTIZE_INT8', False) else 5e-3
assert rel < tol, f'ecart trop grand ({rel:.3f} >= {tol}) — export douteux'

def onnx_forward(ids, am, at):
    return main_sess.run(None, {'input_ids': ids, 'audio_mask': am, 'attention_mask': at})[0]

wav_fr = generate(onnx_forward, TEXT_TEST, REF_TEXT, REF_TOKENS, lang='fr')
sf.write(os.path.join(WORK, 'narrateur_onnx_fr.wav'), wav_fr, 24000)
print('generation ONNX fr terminee,', f'{len(wav_fr)/24000:.1f}s')
ipd.display(ipd.Audio(wav_fr, rate=24000))

In [ ]:
import IPython.display as ipd

# Variante en (langue envoyee par la detection Android actuelle) + comparaison
wav_en = generate(onnx_forward, TEXT_TEST, REF_TEXT, REF_TOKENS, lang='en')
sf.write(os.path.join(WORK, 'narrateur_onnx_en.wav'), wav_en, 24000)
ipd.display(ipd.Audio(wav_en, rate=24000))

n = min(len(wav_fr), len(wav_A)) or 1
cos = float(np.dot(wav_fr[:n], wav_A[:n]) /
           (np.linalg.norm(wav_fr[:n]) * np.linalg.norm(wav_A[:n]) + 1e-9))
print(f'similarite cosinus ONNX(fr) vs PyTorch(fr, denoise) : {cos:.3f} '
      '(>=0.95 attendu : meme graine, meme pipeline)')


## 8. Copie dans le Drive

Le zip `vocoloco_narrateur.zip` contient les DEUX fichiers qui remplacent le backbone
de l'app (~2,4 Go). Le décodeur, l'encodeur, le tokenizer et la config de l'app
restent ceux d'origine.

In [ ]:
import zipfile
marker = os.path.join(EXPORT_DIR, 'README_integration.txt')
open(marker, 'w').write(
    'Integration NekoSpeak (moteur vocoloco)\n'
    '=======================================\n\n'
    'Remplacer dans le modele omnivoice de l app (ModelRepository.kt) :\n'
    ' - omnivoice-main-split.onnx  <- ce graphe\n'
    ' - omnivoice-main.onnx_data   <- ces donnees externes\n'
    'Fichiers inchanges (toujours Gigsu/vocoloco-onnx) :\n'
    ' - omnivoice-decoder.onnx, omnivoice-encoder-fixed.onnx,\n'
    '   omnivoice-config.json, tokenizer\n\n'
    'Voix narrateur = clonage de ref_narrateur.wav (clip_0733)\n'
    f'Texte de reference : {REF_TEXT}\n')

zpath = os.path.join(DRIVE_OUT, 'vocoloco_narrateur.zip')
with zipfile.ZipFile(zpath, 'w', zipfile.ZIP_STORED) as z:
    for f in ['omnivoice-main-split.onnx', 'omnivoice-main.onnx_data', 'README_integration.txt']:
        z.write(os.path.join(EXPORT_DIR, f), f)
print('OK :', zpath, f'({os.path.getsize(zpath)/1e9:.2f} Go)')
print('Contenu Drive/omnivoice :')
for f in sorted(os.listdir(DRIVE_OUT)):
    p = os.path.join(DRIVE_OUT, f)
    print(f'  {f:34s}', f'{os.path.getsize(p)/1e6:8.1f} Mo' if os.path.isfile(p) else '<dossier>')

## 9. Upload vers Hugging Face (structure dossiers par voix)

L'app découvre les voix automatiquement : elle liste les **dossiers** du repo via
l'API tree, puis lit le `voice.json` de chacun. Une nouvelle voix = un nouveau
dossier (graph + poids + ref.wav + voice.json), sans toucher à l'app.

Cette cellule crée le dossier `narrateur/` avec le manifeste complet (refText
exact de la référence, files, dataMinBytes). Crée un token « write » sur
huggingface.co/settings/tokens puis colle-le ci-dessous. **Garde le repo privé**
(poids OmniVoice = CC BY-NC).

⚠️ Ré-exécute cette cellule autant de fois que nécessaire : `upload_folder`
reprend les uploads LFS là où ils se sont arrêtés. Le token reste à saisir dans
Réglages de NekoSpeak (accès read suffit côté app).

In [ ]:
HF_TOKEN = ''   # <--- colle ton token HF write ici
HF_REPO  = 'pruvost/narrateur-vocoloco'
VOICE_ID = 'narrateur'        # nom du dossier dans le repo = id de la voix
VOICE_NAME = 'Narrateur'
VOICE_DESC = 'Voix masculine française dark fantasy (LoRA fine-tune OmniVoice)'
PURGE_ROOT = True             # retire les anciens fichiers posés à plat à la racine

import json, shutil

# --- secours : si la VM Colab a redemarre, EXPORT_DIR est vide -> on reextrait
#     le zip posse sur Mon Drive/omnivoice/ (cellule 22) ---
if not os.path.exists(os.path.join(EXPORT_DIR, 'omnivoice-main.onnx_data')):
    zpath = os.path.join(DRIVE_OUT, 'vocoloco_narrateur.zip')
    assert os.path.exists(zpath), 'run la cellule 2 (setup) puis la cellule 22, ou relance l export'
    import zipfile
    os.makedirs(EXPORT_DIR, exist_ok=True)
    with zipfile.ZipFile(zpath) as z:
        z.extractall(EXPORT_DIR)
    print('EXPORT_DIR restaure depuis', zpath)

if not HF_TOKEN.strip():
    print('Pas de token HF : étape sautée. Le zip Drive suffit pour sideload (adb).')
else:
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN.strip())
    api.create_repo(repo_id=HF_REPO, repo_type='model', private=True, exist_ok=True)

    # Staging local : une voix = un dossier avec son voice.json
    stage = os.path.join(WORK, 'hf_upload', VOICE_ID)
    os.makedirs(stage, exist_ok=True)
    for f in ['omnivoice-main-split.onnx', 'omnivoice-main.onnx_data', 'README_integration.txt']:
        shutil.copy2(os.path.join(EXPORT_DIR, f), stage)
    shutil.copy2(REF_WAV, os.path.join(stage, 'ref.wav'))

    data_bytes = os.path.getsize(os.path.join(stage, 'omnivoice-main.onnx_data'))
    manifest = {
        'id': VOICE_ID,
        'name': VOICE_NAME,
        'description': VOICE_DESC,
        'language': 'fr',
        'gender': 'Male',
        'refText': REF_TEXT,
        'files': {
            'graph': 'omnivoice-main-split.onnx',
            'data': 'omnivoice-main.onnx_data',
            'refAudio': 'ref.wav',
        },
        'dataMinBytes': data_bytes,
    }
    with open(os.path.join(stage, 'voice.json'), 'w', encoding='utf-8') as f:
        json.dump(manifest, f, ensure_ascii=False, indent=2)

    print(f'Contenu de {VOICE_ID}/ :')
    for f in sorted(os.listdir(stage)):
        print(f'  {f:35s} {os.path.getsize(os.path.join(stage, f))/1e6:10.1f} Mo')

    # Upload du dossier complet (LFS géré, reprendre = relancer la cellule)
    api.upload_folder(repo_id=HF_REPO, repo_type='model',
                      folder_path=stage, path_in_repo=VOICE_ID)
    print(f'\nDépôt : https://huggingface.co/{HF_REPO}/tree/main/{VOICE_ID}')
    print("L'app découvre cette voix automatiquement (API tree + voice.json).")

    # Nettoyage des anciens fichiers à plat à la racine (ignorés par l'app)
    if PURGE_ROOT:
        for f in ['omnivoice-main-split.onnx', 'omnivoice-main.onnx_data', 'README_integration.txt']:
            try:
                api.delete_file(path_in_repo=f, repo_id=HF_REPO, repo_type='model')
                print('racine : supprimé', f)
            except Exception as e:
                pass  # fichier absent : rien à faire


## Prochaines étapes

**La branche Kotlin `feat/omnivoice-narrateur` est prête** (3 commits déjà poussés) :
- tenseurs aux rangs exacts dans OmniVoiceEngine.kt (5 appels corrigés),
- URLs du backbone vers `pruvost/narrateur-vocoloco` + token HF dans ModelRepository.kt,
- référence vocale clip_0733 embarquée en asset : la voix « narrateur » se crée
  toute seule au premier démarrage.

**Ton parcours après ce notebook :**
1. écouter `narrateur_onnx_fr.wav` et `narrateur_onnx_en.wav` (l'app envoie
   actuellement `en` pour le français : si `fr` sonne mieux, je patche la
   détection de langue) ;
2. coller un token HF « write » dans la cellule 9 et la relancer -> le repo
   privé est créé et rempli ;
3. coller le token HF « read » dans `NARRATEUR_HF_TOKEN` (ModelRepository.kt) ;
4. compiler la branche et lancer l'app : téléchargement 3,2 Go, puis la voix
   narrateur apparaît dans la liste.

Rappels : garde l'onglet ouvert pendant les cellules longues ; poids OmniVoice
CC BY-NC (repo HF privé) ; token GitHub à expirer en fin de projet.